# 01 — Auditoria do dataset

Verifica o dataset gerado por `python -m cerrado_flora.data build` (ou recriado com `fetch`):
funil de seleção, validação das espécies, distribuição das classes por divisão, licenças,
sobreposição de observadores entre divisões e uma amostra das imagens.

Os manifestos vêm de `data/manifests/` (versionados). As imagens podem estar:
- **localmente**: `outputs/dataset/` (saída do `build`/`fetch`);
- **no Colab**: rode antes o `00_setup_colab.ipynb` e descompacte `<drive_root>/processed/cerrado_df_v1.zip` em `/content/cache/`.

A primeira célula de código detecta o ambiente e ajusta os caminhos.

In [ ]:
import importlib
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:  # rode antes o 00_setup_colab.ipynb (clone/upload do código e montagem do Drive)
    REPO_DIR = Path("/content/ProjetoFinalRNAP_UnB2026")
    IMAGES_DIR = Path("/content/cache/cerrado_df_v1")
else:
    REPO_DIR = Path("..").resolve()
    IMAGES_DIR = REPO_DIR / "outputs/dataset"
MANIFESTS = REPO_DIR / "data/manifests"

# Mesma correção do notebook 00: o kernel em execução não lê o .pth da instalação
# editável, então src/ é adicionado ao caminho explicitamente.
SRC_DIR = str(REPO_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
importlib.invalidate_caches()

import cerrado_flora

assert cerrado_flora.__file__.startswith(SRC_DIR), cerrado_flora.__file__
print("cerrado_flora em", cerrado_flora.__file__, "| imagens em", IMAGES_DIR)

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from cerrado_flora.data.split import check_leakage

images = pd.read_csv(MANIFESTS / "images.csv")
species = pd.read_csv(MANIFESTS / "species.csv")
overlap = pd.read_csv(MANIFESTS / "observer_overlap.csv")
info = json.loads((MANIFESTS / "build_info.json").read_text(encoding="utf-8"))

check_leakage(images)  # falha se houver vazamento ou classe ausente de alguma divisão
print("consulta iNaturalist:", info["inat_query"]["query_utc"])
print("commit:", info["codigo"]["git_commit"], "| alterações não commitadas:", info["codigo"]["git_dirty"])
pd.Series(info["funil"], name="funil").to_frame()

## 1. Validação das espécies (Flora e Funga do Brasil)

In [ ]:
print(species["status"].value_counts().to_string())
cols = ["scientific_name", "n_obs_elegiveis", "match", "ffb_name", "establishment", "domains", "status", "note"]
species.loc[species["status"] != "incluida", cols]

Espécies incluídas, com observações por divisão:

In [ ]:
inc = species[species["status"] == "incluida"].sort_values("class_index")
inc[["class_index", "scientific_name", "family", "endemism", "occurs_df",
     "n_obs_elegiveis", "n_obs_usadas", "n_obs_train", "n_obs_val", "n_obs_test"]]

## 2. Distribuição das classes por divisão

In [ ]:
SPLIT_COLORS = {"train": "#2a78d6", "val": "#eb6834", "test": "#1baf7a"}  # paleta categórica validada
obs = images.drop_duplicates("observation_id")
counts = (obs.pivot_table(index="scientific_name", columns="split", values="observation_id",
                          aggfunc="count")[list(SPLIT_COLORS)].sort_values("train"))

fig, ax = plt.subplots(figsize=(7, 0.22 * len(counts) + 1))
left = 0
for s, c in SPLIT_COLORS.items():
    ax.barh(counts.index, counts[s], left=left, color=c, label=s, height=0.7,
            edgecolor="white", linewidth=1)
    left = left + counts[s]
ax.set_xlabel("observações")
ax.tick_params(axis="y", labelsize=7)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.grid(axis="x", color="#e5e4e0", linewidth=0.6)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False)
ax.set_title("Observações por espécie e divisão", loc="left")
plt.tight_layout()
plt.show()

In [ ]:
summary = pd.DataFrame(info["por_divisao"]).T
summary["fotos_por_obs"] = (summary["fotos"] / summary["observacoes"]).round(2)
summary

## 3. Licenças e atribuição

In [ ]:
pd.crosstab(images["license"], images["split"], margins=True)

In [ ]:
assert images["attribution"].str.len().gt(0).all(), "foto sem atribuição"
images[["photo_id", "license", "attribution", "observation_url"]].sample(5, random_state=0)

## 4. Sobreposição de observadores

A divisão é por observação, não por observador. Fração das observações de validação/teste
cujo observador também contribuiu para o treino da mesma espécie (relatado, não imposto).

In [ ]:
print(info["sobreposicao_observador_media"])
overlap.sort_values("test_obs_observer_in_train", ascending=False).head(10)

In [ ]:
top = (obs.groupby("scientific_name")["observer"]
          .agg(lambda s: s.value_counts(normalize=True).iloc[0]).rename("fracao_maior_observador"))
top.sort_values(ascending=False).head(10).to_frame()

## 5. Fotos descartadas e quase-duplicatas

In [ ]:
dropped = pd.DataFrame(info["fotos_descartadas"])
print("grupos de quase-duplicatas com >1 observação:", info["grupos_com_mais_de_uma_observacao"])
dropped["motivo"].value_counts() if len(dropped) else "nenhuma foto descartada"

## 6. Tamanho das imagens e amostra

In [ ]:
images[["width", "height"]].describe().round(0)

In [ ]:
sample = (images[images["split"] == "train"].drop_duplicates("class_index")
          .sort_values("class_index"))  # uma foto de treino por classe
ncol = 6
nrow = -(-len(sample) // ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(2.2 * ncol, 2.5 * nrow))
for ax in axes.flat:
    ax.axis("off")
for ax, r in zip(axes.flat, sample.itertuples()):
    with Image.open(IMAGES_DIR / r.file) as im:
        ax.imshow(im)
    ax.set_title(f"{r.scientific_name}\n{r.license}", fontsize=7)
plt.tight_layout()
plt.show()